# PrediRuta: construcción de los datasets finales

Este notebook parte de los **seis CSV relacionales depurados**. 
1. construye un diccionario reproducible para convertir las variables categóricas en códigos numéricos
2. crea los momentos complementarios sin accidente 
3. integra las tablas necesarias para los análisis posteriores

## 0. Configuración

In [22]:
# Librerías y parámetros generales
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.spatial import cKDTree

In [23]:
# Configuración de rutas y carpetas
CARPETA_TABLAS = Path('data_procesada') / 'tablas_finales'
CARPETA_CACHE_CLIMA = Path('data_procesada') / 'cache_clima_hibrido'
CARPETA_SALIDA = Path('data_procesada') / 'modelado'

# Crear carpeta de salida si no existe
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

# Rutas de salida de archivos
RUTA_DICCIONARIO = CARPETA_SALIDA / 'DICCIONARIO_CATEGORIAS.csv'
RUTA_OCURRENCIA = CARPETA_SALIDA / 'DATASET_OCURRENCIA.csv'
RUTA_COMPLETA = CARPETA_SALIDA / 'TABLA_COMPLETA_ACCIDENTES.csv'

## 1. Carga de los CSV separados

Los archivos originales no se sobrescriben. Todas las transformaciones se aplican sobre copias en memoria.

In [24]:
# Archivos de entrada
ARCHIVOS = {
    'ACCIDENTE': 'ACCIDENTE.csv',
    'CLIMA': 'CLIMA.csv',
    'VIA': 'VIA.csv',
    'VEHICULO': 'VEHICULO.csv',
    'ACTOR_VIAL': 'ACTOR_VIAL.csv',
    'CAUSA': 'CAUSA.csv',
}

print('Archivos encontrados:', len(ARCHIVOS))

Archivos encontrados: 6


In [25]:
# Cargar las seis tablas
TABLAS = {}

for tabla, archivo in ARCHIVOS.items():

    ruta = CARPETA_TABLAS / archivo

    if tabla == 'ACCIDENTE':
        TABLAS[tabla] = pd.read_csv(
            ruta,
            encoding='utf-8-sig',
            low_memory=False,
            parse_dates=['FECHA_HORA']
        )

    elif tabla == 'CLIMA':
        TABLAS[tabla] = pd.read_csv(
            ruta,
            encoding='utf-8-sig',
            low_memory=False,
            parse_dates=['FECHA_HORA_CLIMA']
        )

    else:
        TABLAS[tabla] = pd.read_csv(
            ruta,
            encoding='utf-8-sig',
            low_memory=False
        )

# Resumen de tablas cargadas
inventario = pd.DataFrame([
    {
        'TABLA': tabla,
        'FILAS': len(datos),
        'COLUMNAS': len(datos.columns),
        'ARCHIVO': ARCHIVOS[tabla]
    }
    for tabla, datos in TABLAS.items()
])

display(inventario)

,TABLA,FILAS,COLUMNAS,ARCHIVO
0,ACCIDENTE,895346,7,ACCIDENTE.csv
1,CLIMA,325230,11,CLIMA.csv
2,VIA,488413,12,VIA.csv
3,VEHICULO,1669752,4,VEHICULO.csv
4,ACTOR_VIAL,1927486,7,ACTOR_VIAL.csv
5,CAUSA,1283835,6,CAUSA.csv


## 2. Diccionario de variables categóricas

En esta sección se identifican las variables categóricas presentes en cada una de las tablas y se construye un diccionario que permita convertir sus valores a códigos numéricos.

Cada combinación de tabla y columna cuenta con su propio catálogo de categorías, siguiendo estas reglas:

- El código `0` representa los valores `SIN INFORMACION`.
- Las demás categorías se organizan alfabéticamente.
- La codificación comienza en `1` y continúa de forma consecutiva.

Para la tabla de causas se valida la relación entre `CODIGO_CAUSA` y `NOMBRE`. Para el modelado se utiliza `NOMBRE`, porque contiene el significado de la causa. `CODIGO_CAUSA` se usa solamente para revisar la consistencia de los datos y luego se excluye de la codificación.

El diccionario generado se conserva como referencia para interpretar posteriormente los códigos asignados y garantizar que la misma transformación pueda aplicarse de manera consistente en las siguientes etapas del análisis.

In [26]:
# Validación de consistencia entre códigos y nombres en la tabla CAUSA
causas_catalogo = TABLAS['CAUSA'][['CODIGO_CAUSA', 'NOMBRE']].copy()

for columna in ['CODIGO_CAUSA', 'NOMBRE']:
    causas_catalogo[columna] = (
        causas_catalogo[columna]
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )

# Validación de consistencia entre códigos y nombres
pares_causa = causas_catalogo.dropna().drop_duplicates()
nombres_por_codigo = pares_causa.groupby('CODIGO_CAUSA')['NOMBRE'].nunique()
codigos_por_nombre = pares_causa.groupby('NOMBRE')['CODIGO_CAUSA'].nunique()
codigos_inconsistentes = nombres_por_codigo[nombres_por_codigo > 1]

# Validación de consistencia entre las dos columnas
validacion_causas = pd.DataFrame({
    'METRICA': [
        'Códigos distintos',
        'Nombres distintos',
        'Códigos con varios nombres',
        'Nombres asociados a varios códigos'
    ],
    'VALOR': [
        causas_catalogo['CODIGO_CAUSA'].nunique(),
        causas_catalogo['NOMBRE'].nunique(),
        int((nombres_por_codigo > 1).sum()),
        int((codigos_por_nombre > 1).sum())
    ],
})

display(validacion_causas)

,METRICA,VALOR
0,Códigos distintos,131
1,Nombres distintos,120
2,Códigos con varios nombres,0
3,Nombres asociados a varios códigos,3


In [27]:
# Listado de nombres que agrupan más de un código administrativo
nombres_con_varios_codigos = (
    codigos_por_nombre[codigos_por_nombre > 1]
    .rename('CANTIDAD_CODIGOS')
    .reset_index()
    .sort_values('CANTIDAD_CODIGOS', ascending=False)
)

display(nombres_con_varios_codigos)

,NOMBRE,CANTIDAD_CODIGOS
2,SIN INFORMACION,9
0,OTRA,3
1,OTRAS,2


Cada `CODIGO_CAUSA` está asociado a un único valor de `NOMBRE`, por lo que no se identifican inconsistencias entre ambas columnas. Sin embargo, se observa que algunos nombres genéricos pueden estar relacionados con varios códigos administrativos.

Para el modelo se conserva la variable `NOMBRE`, ya que representa de forma directa el significado de la causa y facilita su interpretación. A partir de sus categorías se generan los códigos numéricos definidos en el diccionario.

Por esta razón, `CODIGO_CAUSA` se conserva únicamente como referencia dentro de la información original y no se utiliza como variable predictora en el modelo.

In [28]:
# Variables categóricas de cada tabla
COLUMNAS_CATEGORICAS = {
    'ACCIDENTE': ['CLASE_ACCIDENTE'],
    'CLIMA': [],
    'VIA': [
        'GEOMETRIA_PLANTA','GEOMETRIA_TERRENO','GEOMETRIA_SECCION','SENTIDO_VIA',
        'SUPERFICIE_RODADURA','ESTADO_VIA','CONDICION_VIA',
        'ILUMINACION_ARTIFICIAL','SEMAFORO',
    ],
    'VEHICULO': ['CLASE','SERVICIO'],
    'ACTOR_VIAL': ['CONDICION','ESTADO','GENERO'],
    'CAUSA': ['NOMBRE', 'TIPO'],
}

In [29]:
# Construir diccionario de categorías y mapas de transformación
SIN_INFORMACION = 'SIN INFORMACION'

registros_diccionario = []
MAPAS_CATEGORIAS = {}

for tabla, columnas in COLUMNAS_CATEGORICAS.items():
    for columna in columnas:

        # Limpiar categorías
        valores = (
            TABLAS[tabla][columna]
            .astype('string')
            .str.strip()
            .replace('', pd.NA)
            .fillna(SIN_INFORMACION)
        )

        # Ordenar categorías alfabéticamente
        categorias = sorted(set(valores) - {SIN_INFORMACION})

        # Crear mapa numérico
        mapa = {
            SIN_INFORMACION: 0,
            **{
                categoria: codigo
                for codigo, categoria in enumerate(categorias, start=1)
            }
        }

        MAPAS_CATEGORIAS[(tabla, columna)] = mapa

        # Agregar categorías al diccionario
        registros_diccionario.extend([
            [tabla, columna, codigo, valor]
            for valor, codigo in mapa.items()
        ])

# Consolidar diccionario
DICCIONARIO_CATEGORIAS = (
    pd.DataFrame(
        registros_diccionario,
        columns=['TABLA', 'COLUMNA', 'CODIGO', 'DESCRIPCION']
    )
    .sort_values(['TABLA', 'COLUMNA', 'CODIGO'])
    .reset_index(drop=True)
)

display(DICCIONARIO_CATEGORIAS)

,TABLA,COLUMNA,CODIGO,DESCRIPCION
0,ACCIDENTE,CLASE_ACCIDENTE,0,SIN INFORMACION
1,ACCIDENTE,CLASE_ACCIDENTE,1,ATROPELLO
2,ACCIDENTE,CLASE_ACCIDENTE,2,AUTOLESION
3,ACCIDENTE,CLASE_ACCIDENTE,3,CAIDA DE OCUPANTE
4,ACCIDENTE,CLASE_ACCIDENTE,4,CHOQUE
...,...,...,...,...
232,VIA,SUPERFICIE_RODADURA,3,ASFALTO
233,VIA,SUPERFICIE_RODADURA,4,CONCRETO
234,VIA,SUPERFICIE_RODADURA,5,EMPEDRADO
235,VIA,SUPERFICIE_RODADURA,6,OTRO


In [30]:
# Exportar el diccionario
DICCIONARIO_CATEGORIAS.to_csv(
    RUTA_DICCIONARIO, index=False, encoding='utf-8-sig'
)
print(f'Diccionario exportado: {RUTA_DICCIONARIO}')

Diccionario exportado: data_procesada/modelado/DICCIONARIO_CATEGORIAS.csv


## 3. Transformación numérica de las seis tablas

Para cada una de las seis tablas se crea una copia sobre la cual se realiza la transformación de las variables categóricas, reemplazando sus valores originales por los códigos numéricos definidos previamente en el diccionario.

Las tablas originales se conservan sin modificaciones. Las llaves de identificación, las fechas y las variables que ya son numéricas mantienen sus valores originales.

In [31]:
# Aplicar codificación a las variables categóricas
TABLAS_CODIFICADAS = {tabla: datos.copy() for tabla, datos in TABLAS.items()}

resumen_codificacion = []

for tabla, columnas in COLUMNAS_CATEGORICAS.items():
    for columna in columnas: # Iterar sobre las columnas categóricas de cada tabla
        valores = (
            TABLAS_CODIFICADAS[tabla][columna]
            .astype('string')
            .str.strip()
            .replace('', pd.NA)
            .fillna(SIN_INFORMACION)
        )

        codigos = valores.map(MAPAS_CATEGORIAS[(tabla, columna)])

        TABLAS_CODIFICADAS[tabla][columna] = codigos.astype('int16')

        resumen_codificacion.append([
            tabla,
            columna,
            valores.nunique(),
            codigos.min(),
            codigos.max()
        ])

# Crear un resumen de la codificación aplicada
resumen_codificacion = pd.DataFrame(
    resumen_codificacion,
    columns=[
        'TABLA',
        'COLUMNA',
        'CATEGORIAS',
        'CODIGO_MINIMO',
        'CODIGO_MAXIMO'
    ]
)

display(resumen_codificacion)

,TABLA,COLUMNA,CATEGORIAS,CODIGO_MINIMO,CODIGO_MAXIMO
0,ACCIDENTE,CLASE_ACCIDENTE,8,0,7
1,VIA,GEOMETRIA_PLANTA,7,0,6
2,VIA,GEOMETRIA_TERRENO,7,0,6
3,VIA,GEOMETRIA_SECCION,8,0,7
4,VIA,SENTIDO_VIA,6,0,5
5,VIA,SUPERFICIE_RODADURA,8,0,7
6,VIA,ESTADO_VIA,10,0,9
7,VIA,CONDICION_VIA,9,0,8
8,VIA,ILUMINACION_ARTIFICIAL,3,0,2
9,VIA,SEMAFORO,2,1,2


In [32]:
# Exportar tablas codificadas
for tabla, datos in TABLAS_CODIFICADAS.items():
    datos.to_csv(
        CARPETA_SALIDA / f'{tabla}.csv',
        index=False,
        encoding='utf-8-sig'
    )

print(f'Tablas codificadas exportadas en: {CARPETA_SALIDA}')

Tablas codificadas exportadas en: data_procesada/modelado


## 4. Construcción de la data complementaria

Para entrenar un modelo que estime la ocurrencia de accidentes no es suficiente utilizar únicamente los registros donde sí ocurrió un evento. También es necesario contar con observaciones comparables en las que no se haya registrado un accidente, de manera que el modelo pueda aprender a diferenciar ambos escenarios.

### 4.1 Casos con accidente

La unidad de análisis se define como una **celda espacial en una hora determinada**. Para esto, Bogotá se divide inicialmente en celdas regulares de 500 metros utilizando el sistema de coordenadas `EPSG:3116`.

Esta granularidad se adopta como un punto de partida para mantener un equilibrio entre el nivel de detalle espacial y la cantidad de observaciones disponibles para el entrenamiento del modelo. Posteriormente, su tamaño podrá validarse y ajustarse de acuerdo con la distribución de los accidentes y el desempeño obtenido.

Cada accidente se asigna a la celda espacial y a la hora correspondiente. Cuando se presentan varios accidentes dentro de la misma celda y durante la misma hora, estos se agrupan en una sola observación positiva, indicando que en esa combinación de espacio y tiempo ocurrió al menos un accidente.

Posteriormente, las celdas permitirán asociar las predicciones del modelo con los tramos de las rutas consultadas, de manera que cada tramo pueda recibir un nivel de riesgo de acuerdo con su ubicación, fecha y hora.

In [33]:
# Preparar accidentes para el dataset de ocurrencia
TAMANO_CELDA_METROS = 500

# Construir dataset de ocurrencia de accidentes por celda y hora
accidentes = TABLAS_CODIFICADAS['ACCIDENTE'][
    ['ACCIDENTE_ID', 'FECHA_HORA', 'LATITUD', 'LONGITUD']
].copy()

# Asegurar formato numérico
accidentes[['LATITUD', 'LONGITUD']] = accidentes[
    ['LATITUD', 'LONGITUD']
].apply(pd.to_numeric, errors='coerce')

# Redondear fecha y hora
accidentes['FECHA_HORA'] = accidentes['FECHA_HORA'].dt.floor('h')

# Transformar coordenadas geográficas a coordenadas planas (EPSG:3116)
wgs84_a_bogota = Transformer.from_crs(
    'EPSG:4326',
    'EPSG:3116',
    always_xy=True
)

bogota_a_wgs84 = Transformer.from_crs(
    'EPSG:3116',
    'EPSG:4326',
    always_xy=True
)

x, y = wgs84_a_bogota.transform(
    accidentes['LONGITUD'].to_numpy(),
    accidentes['LATITUD'].to_numpy()
)

# Asignar cada accidente a una celda de 500 m
accidentes['CELDA_X'] = (
    np.floor(np.asarray(x) / TAMANO_CELDA_METROS)
    .astype('int32')
)

accidentes['CELDA_Y'] = (
    np.floor(np.asarray(y) / TAMANO_CELDA_METROS)
    .astype('int32')
)

accidentes['CELDA_ID'] = (
    accidentes['CELDA_X'].astype(str)
    + '_'
    + accidentes['CELDA_Y'].astype(str)
)

# Construir casos positivos únicos por celda y hora
casos = (
    accidentes[
        ['CELDA_ID', 'CELDA_X', 'CELDA_Y', 'FECHA_HORA']
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Obtener coordenadas del centro de cada celda
x_centro = (
    casos['CELDA_X'].to_numpy() + 0.5
) * TAMANO_CELDA_METROS

y_centro = (
    casos['CELDA_Y'].to_numpy() + 0.5
) * TAMANO_CELDA_METROS

lon_centro, lat_centro = bogota_a_wgs84.transform(
    x_centro,
    y_centro
)

casos['LATITUD'] = lat_centro
casos['LONGITUD'] = lon_centro
casos['OCURRIO_ACCIDENTE'] = 1
casos['PAR_ID'] = np.arange(len(casos))

display(casos.head())

print(f'Casos positivos celda-hora: {len(casos):,}')
print(f'Celdas utilizadas: {casos["CELDA_ID"].nunique():,}')

,CELDA_ID,CELDA_X,CELDA_Y,FECHA_HORA,LATITUD,LONGITUD,OCURRIO_ACCIDENTE,PAR_ID
0,1997_2021,1997,2021,2017-06-12 05:00:00,4.693414,-74.088774,1,0
1,2008_2016,2008,2016,2012-12-15 20:00:00,4.670805,-74.039203,1,1
2,1997_2018,1997,2018,2014-10-26 21:00:00,4.679849,-74.088774,1,2
3,1987_2014,1987,2014,2012-10-15 18:00:00,4.661761,-74.133838,1,3
4,1980_1998,1980,1998,2015-05-11 10:00:00,4.589413,-74.165373,1,4


Casos positivos celda-hora: 496,968
Celdas utilizadas: 1,554


### 4.2 Controles sin accidente

Para cada caso positivo se genera una observación de control en la que `OCURRIO_ACCIDENTE = 0`. El objetivo es construir un escenario comparable en el que no se haya registrado un accidente, manteniendo condiciones espaciales y temporales similares a las del caso original.

Cada control conserva:

* La misma celda espacial.
* La misma hora del día.
* El mismo día de la semana.

Para generar el control, la fecha original se desplaza aleatoriamente entre una y ocho semanas completas, manteniéndose dentro del mismo año. De esta manera, se conserva el día de la semana y la hora, pero se utiliza un momento diferente al del accidente original.

Adicionalmente, se valida que la combinación de celda, fecha y hora seleccionada no coincida con un accidente registrado ni con otro control previamente generado.

En los casos positivos, cuando se presentan varios accidentes dentro de la misma celda y durante la misma hora, estos se consolidan en una única observación. Por lo tanto, en esta etapa el objetivo del modelo no es estimar la cantidad de accidentes, sino identificar la ocurrencia de al menos un evento en una combinación determinada de espacio y tiempo.

Para la construcción del dataset se incluyen únicamente variables que pueden conocerse **antes o en el momento de realizar la predicción**. Por esta razón, no se incorporan características que dependan de que el accidente ya haya ocurrido, como la gravedad, las causas identificadas, los vehículos involucrados o las condiciones de los actores viales.

Posteriormente, el dataset podrá enriquecerse con variables disponibles previamente, como características de la vía, condiciones meteorológicas y componentes temporales derivados de la fecha y la hora.

> **Nota metodológica:** al utilizar inicialmente un caso negativo por cada caso positivo, el dataset presenta una distribución aproximadamente balanceada entre ambas clases. Esta proporción facilita el entrenamiento y la comparación inicial de modelos, pero no representa la frecuencia real de ocurrencia de accidentes en Bogotá. Este aspecto deberá considerarse posteriormente al interpretar y calibrar las probabilidades generadas por el modelo.


In [34]:
# Generar un control por cada caso positivo
SEMILLA = 42
MAX_SEMANAS_DESPLAZAMIENTO = 8

# Configurar generador de números aleatorios
rng = np.random.default_rng(SEMILLA)

claves_positivas = set(
    zip(casos['CELDA_ID'], casos['FECHA_HORA'])
)

controles = []
claves_controles = set()

# Generar controles desplazando la fecha de cada caso positivo
for _, caso in casos.iterrows(): # Iterar sobre cada caso positivo
    desplazamientos = rng.permutation(
        np.concatenate([
            np.arange(-MAX_SEMANAS_DESPLAZAMIENTO, 0),
            np.arange(1, MAX_SEMANAS_DESPLAZAMIENTO + 1)
        ])
    )

    for semanas in desplazamientos: # Iterar sobre los desplazamientos de semanas
        fecha_control = caso['FECHA_HORA'] + pd.Timedelta(
            weeks=int(semanas)
        )

        # Mantener el control dentro del mismo año
        if fecha_control.year != caso['FECHA_HORA'].year:
            continue

        clave_control = (caso['CELDA_ID'], fecha_control)

        # Evitar una celda-hora con accidente o ya usada como control
        if clave_control in claves_positivas or clave_control in claves_controles:
            continue
        controles.append({
            'PAR_ID': caso['PAR_ID'],
            'CELDA_ID': caso['CELDA_ID'],
            'CELDA_X': caso['CELDA_X'],
            'CELDA_Y': caso['CELDA_Y'],
            'FECHA_HORA': fecha_control,
            'LATITUD': caso['LATITUD'],
            'LONGITUD': caso['LONGITUD'],
            'OCURRIO_ACCIDENTE': 0
        })
        claves_controles.add(clave_control)
        break

# Consolidar controles en un DataFrame
controles = pd.DataFrame(controles)

print(f'Casos positivos: {len(casos):,}')
print(f'Controles creados: {len(controles):,}')

Casos positivos: 496,968
Controles creados: 496,968


In [35]:
# Unir casos positivos y negativos
columnas_base = [
    'PAR_ID',
    'CELDA_ID',
    'CELDA_X',
    'CELDA_Y',
    'FECHA_HORA',
    'LATITUD',
    'LONGITUD',
    'OCURRIO_ACCIDENTE'
]

# Mantener solo casos que lograron un control válido y único
casos_emparejados = casos[
    casos['PAR_ID'].isin(controles['PAR_ID'])
].copy()

# Consolidar dataset de ocurrencia
dataset_ocurrencia = pd.concat(
    [
        casos_emparejados[columnas_base],
        controles[columnas_base]
    ],
    ignore_index=True
)

display(
    dataset_ocurrencia['OCURRIO_ACCIDENTE']
    .value_counts()
    .rename('FILAS')
)

OCURRIO_ACCIDENTE
1    496968
0    496968
Name: FILAS, dtype: int64

### 4.3 Clima de casos y controles

A cada caso positivo y a cada control se le asignan las condiciones climáticas correspondientes a su propia ubicación, fecha y hora.

Para esto, se utiliza la información del nodo climático disponible más cercano a la celda espacial de cada observación, de acuerdo con la fuente correspondiente al periodo analizado:

* `ERA5` para fechas anteriores a 2017.
* `ECMWF IFS` para fechas desde 2017 en adelante.

Las condiciones climáticas del accidente no se trasladan al control. Cada observación consulta de forma independiente el clima correspondiente a su propio momento, permitiendo comparar casos y controles bajo las condiciones meteorológicas que realmente correspondían a cada fecha y hora.

In [36]:
# Construir el catálogo de cachés climáticos
archivos_cache = sorted(CARPETA_CACHE_CLIMA.glob('clima_*.csv'))

catalogo_nodos = []
for ruta in archivos_cache:
    muestra = pd.read_csv(ruta, nrows=1)
    catalogo_nodos.append([
        ruta.name, muestra.loc[0,'MODELO'],
        float(muestra.loc[0,'LATITUD_CELDA']),
        float(muestra.loc[0,'LONGITUD_CELDA']),
    ])
catalogo_nodos = pd.DataFrame(catalogo_nodos, columns=[
    'ARCHIVO_CACHE','MODELO_CLIMA','LATITUD_NODO','LONGITUD_NODO',
]).drop_duplicates('ARCHIVO_CACHE')

display(catalogo_nodos.groupby('MODELO_CLIMA').size().rename('NODOS').reset_index())

,MODELO_CLIMA,NODOS
0,ecmwf_ifs,15
1,era5,4


In [37]:
# Asignar el nodo climático más cercano según el periodo
CORTE_MODELO = pd.Timestamp('2017-01-01')

dataset_ocurrencia['MODELO_CLIMA'] = np.where(
    dataset_ocurrencia['FECHA_HORA'] < CORTE_MODELO,
    'era5',
    'ecmwf_ifs'
)

# Asignar el archivo de caché climático más cercano a cada celda-hora   
dataset_ocurrencia['ARCHIVO_CACHE'] = pd.NA

# Iterar sobre cada modelo climático y sus nodos
for modelo, nodos in catalogo_nodos.groupby('MODELO_CLIMA'): 
    mascara = dataset_ocurrencia['MODELO_CLIMA'].eq(modelo)
    
    if not mascara.any(): # Si no hay observaciones para este modelo
        continue

    # Coordenadas de los nodos climáticos
    x_nodos, y_nodos = wgs84_a_bogota.transform(
        nodos['LONGITUD_NODO'].to_numpy(),
        nodos['LATITUD_NODO'].to_numpy()
    )

    # Coordenadas de las observaciones
    x_obs, y_obs = wgs84_a_bogota.transform(
        dataset_ocurrencia.loc[mascara, 'LONGITUD'].to_numpy(),
        dataset_ocurrencia.loc[mascara, 'LATITUD'].to_numpy()
    )

    # Buscar el nodo climático más cercano
    arbol = cKDTree(np.column_stack([x_nodos, y_nodos]))

    _, posiciones = arbol.query(
        np.column_stack([x_obs, y_obs]),
        k=1
    )

    dataset_ocurrencia.loc[mascara, 'ARCHIVO_CACHE'] = (
        nodos.iloc[posiciones]['ARCHIVO_CACHE'].to_numpy()
    )

In [38]:
# Recuperar clima para las observaciones requeridas
columnas_cache = [
    'FECHA_HORA_CLIMA',
    'temperature_2m',
    'relative_humidity_2m',
    'apparent_temperature',
    'precipitation',
    'rain',
    'cloud_cover',
    'surface_pressure',
    'wind_speed_10m',
    'wind_direction_10m'
]

bloques_clima = []

# Iterar sobre cada archivo de caché y sus observaciones correspondientes
for archivo, grupo in dataset_ocurrencia.groupby('ARCHIVO_CACHE'):

    clima = pd.read_csv(
        CARPETA_CACHE_CLIMA / archivo,
        usecols=columnas_cache,
        parse_dates=['FECHA_HORA_CLIMA']
    )

    clima = clima[
        clima['FECHA_HORA_CLIMA'].isin(grupo['FECHA_HORA'])
    ].copy()

    clima['ARCHIVO_CACHE'] = archivo
    bloques_clima.append(clima)

clima_requerido = pd.concat(bloques_clima, ignore_index=True)

# Incorporar clima al dataset
dataset_ocurrencia = dataset_ocurrencia.merge(
    clima_requerido,
    left_on=['ARCHIVO_CACHE', 'FECHA_HORA'],
    right_on=['ARCHIVO_CACHE', 'FECHA_HORA_CLIMA'],
    how='left'
)

# Renombrar variables climáticas
renombres_clima = {
    'temperature_2m': 'TEMPERATURA_2M',
    'relative_humidity_2m': 'HUMEDAD_RELATIVA_2M',
    'apparent_temperature': 'SENSACION_TERMICA',
    'precipitation': 'PRECIPITACION',
    'rain': 'LLUVIA',
    'cloud_cover': 'NUBOSIDAD',
    'surface_pressure': 'PRESION_SUPERFICIE',
    'wind_speed_10m': 'VELOCIDAD_VIENTO_10M',
    'wind_direction_10m': 'DIRECCION_VIENTO_10M'
}

dataset_ocurrencia.rename(
    columns=renombres_clima,
    inplace=True
)

variables_clima = list(renombres_clima.values())

# Identificar pares con información climática incompleta
pares_sin_clima = dataset_ocurrencia.loc[
    dataset_ocurrencia[variables_clima].isna().any(axis=1),
    'PAR_ID'
].unique()

# Mantener únicamente pares completos
dataset_ocurrencia = dataset_ocurrencia[
    ~dataset_ocurrencia['PAR_ID'].isin(pares_sin_clima)
].copy()

print(f'Pares finales con clima: {dataset_ocurrencia["PAR_ID"].nunique():,}')

Pares finales con clima: 496,758


### 4.4 Variables finales del modelo de ocurrencia

A partir de la fecha y hora de cada observación se generan las variables temporales que podrán ser utilizadas por el modelo: año, mes, día de la semana, hora y un indicador de fin de semana.

Para representar la ubicación espacial se utilizan `CELDA_X` y `CELDA_Y`, que corresponden a las coordenadas numéricas de la celda asociada a cada observación.

La variable `CELDA_ID` se utiliza únicamente como identificador durante las etapas de construcción y validación de los datos, por lo que no se incluye como variable predictora en el archivo final del modelo.


In [39]:
# Derivar variables temporales
dataset_ocurrencia['ANIO'] = dataset_ocurrencia['FECHA_HORA'].dt.year
dataset_ocurrencia['MES'] = dataset_ocurrencia['FECHA_HORA'].dt.month
dataset_ocurrencia['DIA_SEMANA'] = dataset_ocurrencia['FECHA_HORA'].dt.dayofweek
dataset_ocurrencia['HORA'] = dataset_ocurrencia['FECHA_HORA'].dt.hour
dataset_ocurrencia['FIN_SEMANA'] = (
    dataset_ocurrencia['DIA_SEMANA'].isin([5, 6]).astype(int)
)

# Seleccionar variables para modelado
columnas_ocurrencia = [
    'CELDA_X',
    'CELDA_Y',
    'FECHA_HORA',
    'LATITUD',
    'LONGITUD',
    'ANIO',
    'MES',
    'DIA_SEMANA',
    'HORA',
    'FIN_SEMANA',
    *variables_clima,
    'OCURRIO_ACCIDENTE'
]

DATASET_OCURRENCIA = (
    dataset_ocurrencia[columnas_ocurrencia]
    .sort_values(['FECHA_HORA', 'CELDA_X', 'CELDA_Y'])
    .reset_index(drop=True)
)

display(DATASET_OCURRENCIA.head())

,CELDA_X,CELDA_Y,FECHA_HORA,LATITUD,LONGITUD,ANIO,MES,DIA_SEMANA,HORA,FIN_SEMANA,TEMPERATURA_2M,HUMEDAD_RELATIVA_2M,SENSACION_TERMICA,PRECIPITACION,LLUVIA,NUBOSIDAD,PRESION_SUPERFICIE,VELOCIDAD_VIENTO_10M,DIRECCION_VIENTO_10M,OCURRIO_ACCIDENTE
0,1975,2005,2007-01-01,4.621060,-74.187908,2007,1,0,0,0,13.5,85.0,13.0,0.0,0.0,88.0,755.1,5.9,101.0,0
1,1976,2007,2007-01-01,4.630104,-74.183403,2007,1,0,0,0,13.1,89.0,12.8,0.0,0.0,91.0,756.6,5.1,94.0,1
2,1994,2032,2007-01-01,4.743150,-74.102296,2007,1,0,0,0,11.6,88.0,10.7,0.0,0.0,97.0,739.5,5.7,108.0,1
3,1999,2007,2007-01-01,4.630112,-74.079761,2007,1,0,0,0,11.6,88.0,10.7,0.0,0.0,97.0,739.5,5.7,108.0,1
4,2001,2029,2007-01-01,4.729586,-74.070748,2007,1,0,0,0,11.6,88.0,10.7,0.0,0.0,97.0,739.5,5.7,108.0,0


## 5. Construcción de la tabla completa codificada de accidentes

Se construye una tabla consolidada con una sola fila por accidente, con el objetivo de realizar análisis descriptivos y explorar variables que posteriormente puedan ser útiles para estudiar la severidad de los eventos.

Las tablas `ACCIDENTE`, `CLIMA` y `VIA` se integran directamente a partir de sus llaves de relación. En el caso de `VEHICULO`, `ACTOR_VIAL` y `CAUSA`, un mismo accidente puede estar relacionado con varios registros. Por esta razón, estas tablas se resumen previamente antes de realizar la integración, evitando así la duplicación de accidentes en la tabla final.

> **Nota metodológica:** esta tabla incluye variables que solo se conocen después de que el accidente ha ocurrido, como las causas, los vehículos involucrados o las características de los actores viales. Por esta razón, no debe utilizarse de forma completa para entrenar el modelo de ocurrencia de accidentes. Su uso está orientado principalmente al análisis descriptivo y a posibles análisis posteriores de severidad.

### 5.1 Resumen de vehículos

Para cada accidente se calcula la cantidad total de vehículos involucrados. Adicionalmente, se generan indicadores que permiten identificar los principales tipos de vehículos presentes en cada evento.

Este resumen permite integrar la información de la tabla `VEHICULO` sin generar múltiples filas para un mismo accidente.

In [40]:
# Caracterización de vehículos involucrados en los accidentes
vehiculo_cod = TABLAS_CODIFICADAS['VEHICULO']

def codigos_con_texto(tabla, columna, texto):
    mapa = MAPAS_CATEGORIAS[(tabla, columna)]

    return {
        codigo
        for categoria, codigo in mapa.items()
        if texto.upper() in str(categoria).upper()
    }

vehiculos_aux = vehiculo_cod.assign(
    TIENE_MOTOCICLETA=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'MOTO')).astype(int),

    TIENE_BICICLETA=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'BICI')).astype(int),

    TIENE_AUTOMOVIL=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'AUTOMOVIL')).astype(int),

    TIENE_BUS=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'BUS')).astype(int),

    TIENE_CAMION=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'CAMION')).astype(int)
)

resumen_vehiculos = (
    vehiculos_aux
    .groupby('ACCIDENTE_ID', as_index=False)
    .agg(
        CANTIDAD_VEHICULOS=('VEHICULO_ID', 'size'),
        CLASES_VEHICULO_DISTINTAS=('CLASE', 'nunique'),
        SERVICIOS_VEHICULO_DISTINTOS=('SERVICIO', 'nunique'),
        TIENE_MOTOCICLETA=('TIENE_MOTOCICLETA', 'max'),
        TIENE_BICICLETA=('TIENE_BICICLETA', 'max'),
        TIENE_AUTOMOVIL=('TIENE_AUTOMOVIL', 'max'),
        TIENE_BUS=('TIENE_BUS', 'max'),
        TIENE_CAMION=('TIENE_CAMION', 'max')
    )
)

display(resumen_vehiculos.head())

,ACCIDENTE_ID,CANTIDAD_VEHICULOS,CLASES_VEHICULO_DISTINTAS,SERVICIOS_VEHICULO_DISTINTOS,TIENE_MOTOCICLETA,TIENE_BICICLETA,TIENE_AUTOMOVIL,TIENE_BUS,TIENE_CAMION
0,104,2,2,2,0,0,1,1,0
1,105,2,2,2,0,0,1,1,0
2,106,2,2,1,0,0,0,1,0
3,107,2,2,2,0,0,1,1,0
4,108,2,2,2,0,0,1,1,0


### 5.2 Resumen de actores y causas

La información de los actores viales se resume para cada accidente mediante la cantidad total de personas involucradas, variables relacionadas con la edad e indicadores que permiten identificar su condición y estado dentro del evento.

Por su parte, las causas se resumen a partir de la cantidad de registros asociados a cada accidente y la diversidad de causas identificadas por su nombre.

De esta manera, ambas fuentes pueden integrarse a la tabla consolidada manteniendo una sola fila por accidente y evitando la duplicación de registros.

In [41]:
# Caracterización de actores y causas de los accidentes
actor_cod = TABLAS_CODIFICADAS['ACTOR_VIAL']
causa_cod = TABLAS_CODIFICADAS['CAUSA']

# Resumen de actores por accidente
actor_aux = actor_cod.assign(
    ES_CONDUCTOR=actor_cod['CONDICION']
        .isin(codigos_con_texto('ACTOR_VIAL', 'CONDICION', 'CONDUCTOR')).astype(int),

    ES_PASAJERO=actor_cod['CONDICION']
        .isin(codigos_con_texto('ACTOR_VIAL', 'CONDICION', 'PASAJ')).astype(int),

    ES_PEATON=actor_cod['CONDICION']
        .isin(codigos_con_texto('ACTOR_VIAL', 'CONDICION', 'PEATON')).astype(int),

    ES_HERIDO=actor_cod['ESTADO']
        .isin(codigos_con_texto('ACTOR_VIAL', 'ESTADO', 'HERID')).astype(int),

    ES_MUERTO=actor_cod['ESTADO']
        .isin(codigos_con_texto('ACTOR_VIAL', 'ESTADO', 'MUERT')).astype(int)
)

resumen_actores = (
    actor_aux
    .groupby('ACCIDENTE_ID', as_index=False)
    .agg(
        CANTIDAD_ACTORES=('ACTOR_ID', 'size'),
        EDAD_PROMEDIO=('EDAD', 'mean'),
        EDAD_MINIMA=('EDAD', 'min'),
        EDAD_MAXIMA=('EDAD', 'max'),
        CANTIDAD_CONDUCTORES=('ES_CONDUCTOR', 'sum'),
        CANTIDAD_PASAJEROS=('ES_PASAJERO', 'sum'),
        CANTIDAD_PEATONES=('ES_PEATON', 'sum'),
        CANTIDAD_HERIDOS=('ES_HERIDO', 'sum'),
        CANTIDAD_MUERTOS=('ES_MUERTO', 'sum')
    )
)

# Resumen de causas por accidente
resumen_causas = (
    causa_cod
    .groupby('ACCIDENTE_ID', as_index=False)
    .agg(
        CANTIDAD_CAUSAS=('CAUSA_ID', 'size'),
        CAUSAS_DISTINTAS=('NOMBRE', 'nunique'),
        TIPOS_CAUSA_DISTINTOS=('TIPO', 'nunique')
    )
)

display(resumen_actores.head())
display(resumen_causas.head())

,ACCIDENTE_ID,CANTIDAD_ACTORES,EDAD_PROMEDIO,EDAD_MINIMA,EDAD_MAXIMA,CANTIDAD_CONDUCTORES,CANTIDAD_PASAJEROS,CANTIDAD_PEATONES,CANTIDAD_HERIDOS,CANTIDAD_MUERTOS
0,104,2,34.5,28.0,41.0,2,0,0,0,0
1,105,2,54.5,53.0,56.0,2,0,0,0,0
2,106,2,33.0,23.0,43.0,2,0,0,0,0
3,107,2,33.5,22.0,45.0,2,0,0,0,0
4,108,2,34.5,27.0,42.0,2,0,0,0,0


,ACCIDENTE_ID,CANTIDAD_CAUSAS,CAUSAS_DISTINTAS,TIPOS_CAUSA_DISTINTOS
0,104,1,1,1
1,105,1,1,1
2,106,1,1,1
3,107,1,1,1
4,108,1,1,1


### 5.3 Integración de la tabla completa

Finalmente, se integran las seis fuentes de información para construir una tabla consolidada con una sola fila por cada `ACCIDENTE_ID`.

Durante esta integración, los valores faltantes asociados a variables categóricas codificadas, indicadores y conteos se completan con `0`, manteniendo así una estructura consistente para el análisis posterior.

El resultado corresponde a una tabla completa por accidente, que reúne en un mismo registro la información disponible sobre el evento, el clima, la vía, los vehículos, los actores viales y las causas asociadas.


In [42]:
# Integrar una fila por accidente
accidente_cod = TABLAS_CODIFICADAS['ACCIDENTE']
clima_cod = TABLAS_CODIFICADAS['CLIMA']
via_cod = TABLAS_CODIFICADAS['VIA']

TABLA_COMPLETA_ACCIDENTES = (
    accidente_cod
    .merge(clima_cod, on='CLIMA_ID', how='left', validate='m:1')
    .merge(via_cod, on='ACCIDENTE_ID', how='left', validate='1:1')
    .merge(resumen_vehiculos, on='ACCIDENTE_ID', how='left', validate='1:1')
    .merge(resumen_actores, on='ACCIDENTE_ID', how='left', validate='1:1')
    .merge(resumen_causas, on='ACCIDENTE_ID', how='left', validate='1:1')
)

# Completar categorías sin información
columnas_categoricas = [
    'CLASE_ACCIDENTE',
    'GEOMETRIA_PLANTA',
    'GEOMETRIA_TERRENO',
    'GEOMETRIA_SECCION',
    'SENTIDO_VIA',
    'SUPERFICIE_RODADURA',
    'ESTADO_VIA',
    'CONDICION_VIA',
    'ILUMINACION_ARTIFICIAL',
    'SEMAFORO'
]

TABLA_COMPLETA_ACCIDENTES[columnas_categoricas] = (
    TABLA_COMPLETA_ACCIDENTES[columnas_categoricas]
    .fillna(0)
    .astype(int)
)

# Completar variables de conteo
columnas_conteo = [
    'CANTIDAD_VEHICULOS',
    'CLASES_VEHICULO_DISTINTAS',
    'SERVICIOS_VEHICULO_DISTINTOS',
    'TIENE_MOTOCICLETA',
    'TIENE_BICICLETA',
    'TIENE_AUTOMOVIL',
    'TIENE_BUS',
    'TIENE_CAMION',
    'CANTIDAD_ACTORES',
    'CANTIDAD_CONDUCTORES',
    'CANTIDAD_PASAJEROS',
    'CANTIDAD_PEATONES',
    'CANTIDAD_HERIDOS',
    'CANTIDAD_MUERTOS',
    'CANTIDAD_CAUSAS',
    'CAUSAS_DISTINTAS',
    'TIPOS_CAUSA_DISTINTOS'
]

TABLA_COMPLETA_ACCIDENTES[columnas_conteo] = (
    TABLA_COMPLETA_ACCIDENTES[columnas_conteo]
    .fillna(0)
    .astype(int)
)

display(TABLA_COMPLETA_ACCIDENTES.head())

,ACCIDENTE_ID,FECHA_HORA,LATITUD,LONGITUD,CLASE_ACCIDENTE,OBJETIVO_GRAVE,CLIMA_ID,FECHA_HORA_CLIMA,TEMPERATURA_2M,HUMEDAD_RELATIVA_2M,...,EDAD_MINIMA,EDAD_MAXIMA,CANTIDAD_CONDUCTORES,CANTIDAD_PASAJEROS,CANTIDAD_PEATONES,CANTIDAD_HERIDOS,CANTIDAD_MUERTOS,CANTIDAD_CAUSAS,CAUSAS_DISTINTAS,TIPOS_CAUSA_DISTINTOS
0,4484660,2017-06-12 05:30:00,4.693807,-74.090924,4,0,94817,2017-06-12 06:00:00,10.9,98,...,34.0,70.0,2,0,0,0,0,1,1,1
1,449558,2012-12-15 20:30:00,4.669288,-74.040677,4,0,300449,2012-12-15 21:00:00,12.3,86,...,25.0,46.0,2,0,0,0,0,2,2,1
2,513150,2014-10-26 21:25:00,4.679568,-74.087407,1,1,311806,2014-10-26 21:00:00,11.2,99,...,27.0,36.0,1,0,3,3,0,1,1,1
3,443270,2012-10-15 18:00:00,4.662473,-74.133350,4,0,251355,2012-10-15 18:00:00,14.3,90,...,53.0,70.0,2,0,0,0,0,2,2,1
4,4412699,2015-05-11 10:50:00,4.587187,-74.166937,4,0,193825,2015-05-11 11:00:00,19.7,44,...,24.0,24.0,2,0,0,0,0,1,1,1


## 6. Construcción de los datasets finales para modelado

A partir de las tablas trabajadas, se construyen dos datasets de modelado con objetivos diferentes y se conserva una tercera tabla consolidada para análisis histórico y exploratorio.

La lógica analítica de PrediRuta se organiza alrededor de tres preguntas principales:

1. **Ocurrencia:** ¿qué tan propenso es un sector y un momento determinado a presentar un siniestro?

2. **Tipo de evento:** si ocurre un siniestro, ¿qué clase de accidente podría presentarse?

3. **Severidad:** si ocurre un siniestro, ¿qué tan probable es que corresponda a un evento grave?

Las variables climáticas se incorporan como un grupo adicional de predictores dentro de las mismas observaciones. Esto permitirá comparar el desempeño de cada modelo bajo dos escenarios: **sin variables climáticas** y **con variables climáticas**. De esta manera, será posible evaluar si la información meteorológica aporta capacidad predictiva adicional y decidir posteriormente si debe incorporarse al cálculo final de criticidad.

> **Nota metodológica:** las variables que solo se conocen después de ocurrido el accidente, como los vehículos involucrados, los actores viales, los heridos, los fallecidos o las causas registradas, no se utilizan para predecir la ocurrencia de eventos futuros. Estas variables se conservan en la tabla completa para caracterización, análisis histórico y exploración de posibles relaciones con la severidad.

### 6.1 Dataset de ocurrencia

El primer dataset utiliza como unidad de análisis una combinación **celda-hora** e incluye tanto observaciones en las que ocurrió al menos un accidente como controles comparables en los que no se registró ningún evento.

La variable objetivo es `OCURRIO_ACCIDENTE`:

* `1`: se registró al menos un accidente en la celda durante esa hora.
* `0`: no se registró un accidente en la combinación celda-hora seleccionada como control.

Para este modelo se utilizan únicamente variables que pueden conocerse antes o en el momento de realizar una predicción, como la ubicación, los componentes temporales y las condiciones meteorológicas.

Las variables `PAR_ID` y `CELDA_ID` se conservan para mantener la trazabilidad de las observaciones y facilitar las validaciones posteriores.

Para la experimentación se plantean dos grupos de variables:

* **Modelo base:** variables espaciales y temporales.
* **Modelo con clima:** variables espaciales y temporales más las variables meteorológicas.

La comparación entre ambos modelos debe realizarse utilizando exactamente las mismas observaciones y la misma separación entre los conjuntos de entrenamiento y prueba. De esta manera, cualquier diferencia en el desempeño podrá asociarse de forma más directa al aporte de las variables climáticas.


In [48]:
# Definir grupos de predictores para el modelo de ocurrencia
PREDICTORES_ESPACIOTEMPORALES = [
    'CELDA_X',
    'CELDA_Y',
    'ANIO',
    'MES',
    'DIA_SEMANA',
    'HORA',
    'FIN_SEMANA'
]

PREDICTORES_CLIMA = variables_clima.copy()

# Construir dataset final de ocurrencia
columnas_ocurrencia = [
    'PAR_ID',
    'CELDA_ID',
    'FECHA_HORA',
    'LATITUD',
    'LONGITUD',
    *PREDICTORES_ESPACIOTEMPORALES,
    *PREDICTORES_CLIMA,
    'OCURRIO_ACCIDENTE'
]

DATASET_OCURRENCIA = (
    dataset_ocurrencia[columnas_ocurrencia]
    .sort_values(['FECHA_HORA', 'CELDA_X', 'CELDA_Y'])
    .reset_index(drop=True)
)

display(DATASET_OCURRENCIA)

print(f'Casos con accidente: '
      f'{DATASET_OCURRENCIA["OCURRIO_ACCIDENTE"].eq(1).sum():,}')
print(f'Controles sin accidente: '
      f'{DATASET_OCURRENCIA["OCURRIO_ACCIDENTE"].eq(0).sum():,}')

,PAR_ID,CELDA_ID,FECHA_HORA,LATITUD,LONGITUD,CELDA_X,CELDA_Y,ANIO,MES,DIA_SEMANA,...,TEMPERATURA_2M,HUMEDAD_RELATIVA_2M,SENSACION_TERMICA,PRECIPITACION,LLUVIA,NUBOSIDAD,PRESION_SUPERFICIE,VELOCIDAD_VIENTO_10M,DIRECCION_VIENTO_10M,OCURRIO_ACCIDENTE
0,135390,1975_2005,2007-01-01 00:00:00,4.621060,-74.187908,1975,2005,2007,1,0,...,13.5,85.0,13.0,0.0,0.0,88.0,755.1,5.9,101.0,0
1,113893,1976_2007,2007-01-01 00:00:00,4.630104,-74.183403,1976,2007,2007,1,0,...,13.1,89.0,12.8,0.0,0.0,91.0,756.6,5.1,94.0,1
2,323638,1994_2032,2007-01-01 00:00:00,4.743150,-74.102296,1994,2032,2007,1,0,...,11.6,88.0,10.7,0.0,0.0,97.0,739.5,5.7,108.0,1
3,126027,1999_2007,2007-01-01 00:00:00,4.630112,-74.079761,1999,2007,2007,1,0,...,11.6,88.0,10.7,0.0,0.0,97.0,739.5,5.7,108.0,1
4,7373,2001_2029,2007-01-01 00:00:00,4.729586,-74.070748,2001,2029,2007,1,0,...,11.6,88.0,10.7,0.0,0.0,97.0,739.5,5.7,108.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
993511,495693,1985_2020,2026-08-10 19:00:00,4.688889,-74.142853,1985,2020,2026,8,0,...,15.4,53.0,12.5,0.0,0.0,3.0,760.3,12.6,147.0,0
993512,496943,1988_2019,2026-08-10 19:00:00,4.684369,-74.129333,1988,2019,2026,8,0,...,15.4,53.0,12.5,0.0,0.0,3.0,760.3,12.6,147.0,0
993513,496884,1993_2024,2026-08-11 13:00:00,4.706978,-74.106801,1993,2024,2026,8,1,...,20.2,46.0,19.7,0.1,0.1,34.0,762.3,15.7,146.0,0
993514,495526,1989_2025,2026-08-11 18:00:00,4.711498,-74.124828,1989,2025,2026,8,1,...,14.1,72.0,11.5,0.0,0.0,93.0,758.2,15.9,148.0,0


Casos con accidente: 496,758
Controles sin accidente: 496,758


### 6.2 Dataset de eventos ocurridos

El segundo dataset contiene únicamente accidentes que efectivamente ocurrieron y se utilizará para desarrollar modelos complementarios condicionados a la existencia de un evento.

A partir de este conjunto se podrán analizar dos variables objetivo:

* `CLASE_ACCIDENTE`: permite estimar qué tipo de accidente podría presentarse una vez ocurre un evento.
* `OBJETIVO_GRAVE`: permite estimar la probabilidad de que el accidente corresponda a un evento grave.

In [47]:
# Construir dataset de accidentes ocurridos directamente desde ACCIDENTE + CLIMA
eventos = (
    TABLAS_CODIFICADAS['ACCIDENTE']
    .merge(
        TABLAS_CODIFICADAS['CLIMA'],
        on='CLIMA_ID',
        how='left',
        validate='m:1'
    )
    .copy()
)

# Asignar cada accidente a la misma estructura espacial utilizada en ocurrencia
x_evento, y_evento = wgs84_a_bogota.transform(
    eventos['LONGITUD'].to_numpy(),
    eventos['LATITUD'].to_numpy()
)

eventos['CELDA_X'] = (
    np.floor(np.asarray(x_evento) / TAMANO_CELDA_METROS)
    .astype('int32')
)

eventos['CELDA_Y'] = (
    np.floor(np.asarray(y_evento) / TAMANO_CELDA_METROS)
    .astype('int32')
)

eventos['CELDA_ID'] = (
    eventos['CELDA_X'].astype(str)
    + '_'
    + eventos['CELDA_Y'].astype(str)
)

# Derivar variables temporales
eventos['ANIO'] = eventos['FECHA_HORA'].dt.year
eventos['MES'] = eventos['FECHA_HORA'].dt.month
eventos['DIA_SEMANA'] = eventos['FECHA_HORA'].dt.dayofweek
eventos['HORA'] = eventos['FECHA_HORA'].dt.hour
eventos['FIN_SEMANA'] = (
    eventos['DIA_SEMANA'].isin([5, 6]).astype(int)
)

# Predictores disponibles antes o durante una consulta futura
PREDICTORES_EVENTO = PREDICTORES_ESPACIOTEMPORALES.copy()

columnas_eventos = [
    'ACCIDENTE_ID',
    'CELDA_ID',
    'FECHA_HORA',
    'LATITUD',
    'LONGITUD',
    *PREDICTORES_EVENTO,
    *PREDICTORES_CLIMA,
    'CLASE_ACCIDENTE',
    'OBJETIVO_GRAVE'
]

DATASET_EVENTOS = (
    eventos[columnas_eventos]
    .sort_values('FECHA_HORA')
    .reset_index(drop=True)
)

display(DATASET_EVENTOS)

,ACCIDENTE_ID,CELDA_ID,FECHA_HORA,LATITUD,LONGITUD,CELDA_X,CELDA_Y,ANIO,MES,DIA_SEMANA,...,HUMEDAD_RELATIVA_2M,SENSACION_TERMICA,PRECIPITACION,LLUVIA,NUBOSIDAD,PRESION_SUPERFICIE,VELOCIDAD_VIENTO_10M,DIRECCION_VIENTO_10M,CLASE_ACCIDENTE,OBJETIVO_GRAVE
0,12798,1999_2007,2007-01-01 00:00:00,4.629939,-74.080151,1999,2007,2007,1,0,...,88,10.7,0.0,0.0,97,739.5,5.7,108,7,1
1,10819805,1999_2007,2007-01-01 00:00:00,4.629939,-74.080151,1999,2007,2007,1,0,...,88,10.7,0.0,0.0,97,739.5,5.7,108,7,1
2,13707,1994_2032,2007-01-01 00:45:00,4.742590,-74.100732,1994,2032,2007,1,0,...,89,9.2,0.0,0.0,95,738.2,6.0,107,4,0
3,10819463,1994_2032,2007-01-01 00:45:00,4.742590,-74.100732,1994,2032,2007,1,0,...,89,9.2,0.0,0.0,95,738.2,6.0,107,4,0
4,10861126,1976_2007,2007-01-01 00:50:00,4.628402,-74.184284,1976,2007,2007,1,0,...,88,11.3,0.0,0.0,89,755.4,5.4,94,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895341,11345986,1979_1995,2026-08-09 20:00:00,4.577361,-74.170757,1979,1995,2026,8,6,...,49,13.4,0.0,0.0,27,760.5,10.4,116,4,1
895342,11345978,2010_2021,2026-08-10 08:26:00,4.692480,-74.029778,2010,2021,2026,8,0,...,73,11.4,0.0,0.0,27,754.3,8.2,157,1,1
895343,11345977,1986_1991,2026-08-10 14:50:00,4.557907,-74.138602,1986,1991,2026,8,0,...,54,14.3,0.0,0.0,9,753.5,20.8,128,1,1
895344,11346001,1991_2009,2026-08-12 13:20:00,4.639754,-74.116059,1991,2009,2026,8,2,...,40,19.2,0.0,0.0,100,761.3,14.1,153,4,1


### 6.3 Estructura de experimentación

Los datasets construidos permiten plantear tres objetivos predictivos diferentes sin duplicar innecesariamente la información.

| Componente     | Dataset              | Variable objetivo   | Comparación propuesta                     |
| -------------- | -------------------- | ------------------- | ----------------------------------------- |
| Ocurrencia     | `DATASET_OCURRENCIA` | `OCURRIO_ACCIDENTE` | Espacio-tiempo vs. espacio-tiempo + clima |
| Tipo de evento | `DATASET_EVENTOS`    | `CLASE_ACCIDENTE`   | Espacio-tiempo vs. espacio-tiempo + clima |
| Severidad      | `DATASET_EVENTOS`    | `OBJETIVO_GRAVE`    | Espacio-tiempo vs. espacio-tiempo + clima |

Las variables meteorológicas se manejan como un **bloque adicional de predictores**. Para cada uno de los tres objetivos se plantea entrenar primero un modelo base utilizando únicamente variables espaciales y temporales y, posteriormente, repetir el experimento incorporando las variables climáticas.


In [45]:
# Resumen de los datasets disponibles para la etapa de modelado
resumen_modelado = pd.DataFrame([
    {
        'DATASET': 'DATASET_OCURRENCIA',
        'UNIDAD_ANALISIS': 'Celda-hora',
        'FILAS': len(DATASET_OCURRENCIA),
        'COLUMNAS': len(DATASET_OCURRENCIA.columns),
        'OBJETIVO': 'OCURRIO_ACCIDENTE'
    },
    {
        'DATASET': 'DATASET_EVENTOS',
        'UNIDAD_ANALISIS': 'Accidente',
        'FILAS': len(DATASET_EVENTOS),
        'COLUMNAS': len(DATASET_EVENTOS.columns),
        'OBJETIVO': 'CLASE_ACCIDENTE / OBJETIVO_GRAVE'
    },
    {
        'DATASET': 'TABLA_COMPLETA_ACCIDENTES',
        'UNIDAD_ANALISIS': 'Accidente',
        'FILAS': len(TABLA_COMPLETA_ACCIDENTES),
        'COLUMNAS': len(TABLA_COMPLETA_ACCIDENTES.columns),
        'OBJETIVO': 'Análisis histórico y exploratorio'
    }
])

display(resumen_modelado)

,DATASET,UNIDAD_ANALISIS,FILAS,COLUMNAS,OBJETIVO
0,DATASET_OCURRENCIA,Celda-hora,993516,22,OCURRIO_ACCIDENTE
1,DATASET_EVENTOS,Accidente,895346,23,CLASE_ACCIDENTE / OBJETIVO_GRAVE
2,TABLA_COMPLETA_ACCIDENTES,Accidente,895346,48,Análisis histórico y exploratorio


### 6.4 Tabla completa de accidentes

`TABLA_COMPLETA_ACCIDENTES` se conserva como la tabla con mayor nivel de detalle de los eventos. Contiene una fila por accidente e integra la información disponible de las fuentes de accidente, clima, vía, vehículos, actores viales y causas.

In [52]:
# Verificar la estructura de la tabla completa de accidentes
display(TABLA_COMPLETA_ACCIDENTES)


,ACCIDENTE_ID,FECHA_HORA,LATITUD,LONGITUD,CLASE_ACCIDENTE,OBJETIVO_GRAVE,CLIMA_ID,FECHA_HORA_CLIMA,TEMPERATURA_2M,HUMEDAD_RELATIVA_2M,...,EDAD_MINIMA,EDAD_MAXIMA,CANTIDAD_CONDUCTORES,CANTIDAD_PASAJEROS,CANTIDAD_PEATONES,CANTIDAD_HERIDOS,CANTIDAD_MUERTOS,CANTIDAD_CAUSAS,CAUSAS_DISTINTAS,TIPOS_CAUSA_DISTINTOS
0,4484660,2017-06-12 05:30:00,4.693807,-74.090924,4,0,94817,2017-06-12 06:00:00,10.9,98,...,34.0,70.0,2,0,0,0,0,1,1,1
1,449558,2012-12-15 20:30:00,4.669288,-74.040677,4,0,300449,2012-12-15 21:00:00,12.3,86,...,25.0,46.0,2,0,0,0,0,2,2,1
2,513150,2014-10-26 21:25:00,4.679568,-74.087407,1,1,311806,2014-10-26 21:00:00,11.2,99,...,27.0,36.0,1,0,3,3,0,1,1,1
3,443270,2012-10-15 18:00:00,4.662473,-74.133350,4,0,251355,2012-10-15 18:00:00,14.3,90,...,53.0,70.0,2,0,0,0,0,2,2,1
4,4412699,2015-05-11 10:50:00,4.587187,-74.166937,4,0,193825,2015-05-11 11:00:00,19.7,44,...,24.0,24.0,2,0,0,0,0,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895341,11344668,2026-06-02 07:50:00,4.695635,-74.087662,1,1,122893,2026-06-02 08:00:00,15.4,71,...,NaN,NaN,0,0,0,0,0,0,0,0
895342,11343968,2026-05-27 23:30:00,4.731127,-74.032113,4,1,169027,2026-05-28 00:00:00,13.2,99,...,NaN,NaN,0,0,0,0,0,0,0,0
895343,11345945,2026-08-05 20:38:00,4.633822,-74.076292,1,1,86297,2026-08-05 21:00:00,7.6,94,...,NaN,NaN,0,0,0,0,0,0,0,0
895344,11345008,2026-06-12 01:00:00,4.689477,-74.118621,4,1,122942,2026-06-12 01:00:00,13.4,85,...,NaN,NaN,0,0,0,0,0,0,0,0


## 7. Exportación

Como resultado del procesamiento se exportan cuatro productos finales:

1. `DICCIONARIO_CATEGORIAS.csv`: contiene la codificación utilizada para transformar las variables categóricas y permite interpretar posteriormente los códigos asignados.

2. `DATASET_OCURRENCIA.csv`: corresponde al dataset principal para modelar la ocurrencia de siniestros a partir de observaciones celda-hora con y sin accidente.

3. `DATASET_EVENTOS.csv`: contiene únicamente accidentes ocurridos y se utiliza para modelar el tipo de evento y la severidad, condicionados a que exista un siniestro.

4. `TABLA_COMPLETA_ACCIDENTES.csv`: reúne la información consolidada por accidente y se conserva para análisis histórico, exploración de relaciones y evaluación posterior de posibles variables.

Los modelos con y sin clima se construirán posteriormente a partir de estos mismos datasets, modificando únicamente los grupos de predictores utilizados en cada experimento. De esta manera, será posible comparar de forma consistente el aporte adicional de las variables meteorológicas.


In [46]:
# Ruta adicional para el dataset de eventos
RUTA_EVENTOS = CARPETA_SALIDA / 'DATASET_EVENTOS.csv'

# Exportar productos finales
DICCIONARIO_CATEGORIAS.to_csv(
    RUTA_DICCIONARIO,
    index=False,
    encoding='utf-8-sig'
)

DATASET_OCURRENCIA.to_csv(
    RUTA_OCURRENCIA,
    index=False,
    encoding='utf-8-sig'
)

DATASET_EVENTOS.to_csv(
    RUTA_EVENTOS,
    index=False,
    encoding='utf-8-sig'
)

TABLA_COMPLETA_ACCIDENTES.to_csv(
    RUTA_COMPLETA,
    index=False,
    encoding='utf-8-sig'
)

print('Archivos generados:')
print(f'- Diccionario:    {DICCIONARIO_CATEGORIAS.shape}')
print(f'- Ocurrencia:     {DATASET_OCURRENCIA.shape}')
print(f'- Eventos:        {DATASET_EVENTOS.shape}')
print(f'- Tabla completa: {TABLA_COMPLETA_ACCIDENTES.shape}')

Archivos generados:
- Diccionario:    (237, 4)
- Ocurrencia:     (993516, 22)
- Eventos:        (895346, 23)
- Tabla completa: (895346, 48)
